# Chapter 09 Companion Notebook: KNN: Credit Card Fraud Detection

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch09_KNN_Credit_Card_Fraud.ipynb)

This notebook accompanies Chapter 09 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Credit card fraud: KNN

### Use "Credit card fraud.csv".

- dist_from_home: The distance from the cardholder's home to the transaction location.
- dist_from_last_transaction: The distance from the last transaction.  
- ratio_to_median_price: The ratio of the transaction amount to the median purchase price.
- repeat_retailer: whether the transaction was made at the same retailer (1 for same retailer, 0 for different retailer).  
- used_chip: whether the transaction was processed using a chip-enabled credit card (1 for chip used, 0 for not used).  
- used_pin: whether the transaction was authorized using a PIN number (1 for PIN used, 0 for not used).
- online_order: whether the transaction was made online (1 for online, 0 for in-person).  
- fraud: whether the transaction was identified as fraudulent (1 for fraud, 0 for not fraud).

In [ ]:
# Suppress warnings

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('Credit card fraud.csv')
df.head()

### 1. Define dependent and independent variables. Split the data into training and test sets (random_state=10). Then, standardize the continuous variables only.
- Dependent variable: 'fraud'
- Independent variable: All other variables.

In [ ]:
y = df['fraud']

# Separate continuous and categorical variables
x1 = df[['dist_from_home', 'dist_from_last_transaction', 'ratio_to_median_price']]  # continuous variables
x2 = df[['repeat_retailer', 'used_chip', 'used_pin', 'online_order']]  # categorical variables

# Split data into training and testing sets
x1_train, x1_test, x2_train, x2_test, ytrain, ytest = train_test_split(x1, x2, y, random_state=10)

# Standardize only the continuous variables
scaler = StandardScaler()
x1_train = scaler.fit_transform(x1_train)
x1_test = scaler.transform(x1_test)

# Combine standardized continuous variables with categorical variables
xtrain = np.hstack((x1_train, x2_train))
xtest = np.hstack((x1_test, x2_test))

### 2. Run a KNN analysis with k=1~10. Visualize training and test accuracy for each of k.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Range of k values to test
kval = range(1, 11)
train_accuracy = []
test_accuracy = []

for k in kval:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(xtrain, ytrain)

    # Compute accuracy for training and testing sets
    train_acc = accuracy_score(ytrain, knn.predict(xtrain))
    test_acc = accuracy_score(ytest, knn.predict(xtest))
    train_accuracy.append(train_acc)
    test_accuracy.append(test_acc)

# Plot the results
plt.plot(kval, train_accuracy, marker='o', label='Training Accuracy')
plt.plot(kval, test_accuracy, marker='x', label='Testing Accuracy')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Accuracy')
plt.legend()

### 5.	Run KNN with the optimal k value and report confusion matrix on the test data.

In [ ]:
from sklearn.metrics import confusion_matrix

# Find the optimal k (highest test accuracy)
optimal_k = kval[test_accuracy.index(max(test_accuracy))]

# Train KNN with optimal k
knn_optimal = KNeighborsClassifier(n_neighbors=optimal_k)
knn_optimal.fit(xtrain, ytrain)

# Predict on test data
ypred = knn_optimal.predict(xtest)

# Compute and display confusion matrix
print("Optimal K =", optimal_k)
confusion_matrix(ytest, ypred)

|          | Predicted 0   | Predicted 1 |
|----------|:-------------:|------------:|
| Actual 0 |       TN      |      FP     |
| Actual 1 |       FN      |      TP     |

In [ ]:
# Visualize confusion matrix

import seaborn as sns
conf = confusion_matrix(ytest, ypred)
sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Fraud", "Fraud"], yticklabels=["Non-Fraud", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("True")